# NRC-VAD Arousal Intensity

This notebook estimates annual affective intensity for ADHD, Autism, and the three baseline terms. It reuses the NRC-VAD collocate handoff built in the Sentiment notebook and computes Baes-style annual arousal indices from local target-window collocates.


## Setup

The diachronic axis is publication year (`lsc_year`). The notebook reads matched NRC-VAD collocates from `data/interim/lsc/vad`, computes annual arousal scores with document-level bootstrap confidence intervals, and writes report-ready outputs under `data/processed/lsc/intensity` and `reports/figures/lsc/intensity`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#263238",
        "axes.labelcolor": "#263238",
        "axes.titlecolor": "#263238",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#D7DEE2",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "legend.frameon": False,
        "xtick.color": "#263238",
        "ytick.color": "#263238",
    }
)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
VAD_MATCH_PATH = PROJECT_ROOT / "data/interim/lsc/vad/lsc_vad_collocate_matches.parquet"
VAD_COVERAGE_PATH = PROJECT_ROOT / "data/interim/lsc/vad/lsc_vad_context_coverage.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/lsc/intensity"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/intensity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_YEARS = list(range(2014, 2027))
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_UNITS = TARGET_UNITS + BASELINE_UNITS
UNIT_COLOURS = {
    "ADHD": "#0072B2",
    "Autism": "#D55E00",
    "frustration": "#009E73",
    "loneliness": "#CC79A7",
    "sadness": "#6E6E6E",
}
UNIT_MARKERS = {
    "ADHD": "o",
    "Autism": "s",
    "frustration": "^",
    "loneliness": "D",
    "sadness": "v",
}
BOOTSTRAP_REPETITIONS = 500
RANDOM_SEED = 123


## Load VAD Handoff

The VAD handoff contains one row per matched collocate occurrence. This notebook uses the same preprocessing decisions as Sentiment: local ±5-token windows, lemmatised NRC-VAD matching, focal-term exclusion only, and stopwords retained for Baes comparability.


In [ ]:
if not VAD_MATCH_PATH.exists():
    raise FileNotFoundError(f"Missing VAD match handoff: {VAD_MATCH_PATH}")
if not VAD_COVERAGE_PATH.exists():
    raise FileNotFoundError(f"Missing VAD coverage handoff: {VAD_COVERAGE_PATH}")

vad_matches = pd.read_parquet(VAD_MATCH_PATH)
context_coverage = pd.read_parquet(VAD_COVERAGE_PATH)

required_match_columns = {
    "context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "registered_domain",
    "collocate",
    "arousal",
}
required_coverage_columns = {
    "context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "candidate_collocate_tokens",
    "matched_vad_units",
    "matched_token_positions",
    "has_vad_match",
}
missing_match_columns = sorted(required_match_columns - set(vad_matches.columns))
missing_coverage_columns = sorted(required_coverage_columns - set(context_coverage.columns))
if missing_match_columns:
    raise RuntimeError(f"VAD match handoff is missing columns: {missing_match_columns}")
if missing_coverage_columns:
    raise RuntimeError(f"VAD coverage handoff is missing columns: {missing_coverage_columns}")

observed_units = sorted(context_coverage["analysis_unit"].dropna().unique())
observed_years = sorted(context_coverage["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units in VAD coverage handoff: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years in VAD coverage handoff: {missing_years}")

handoff_summary = pd.DataFrame(
    {
        "metric": [
            "matched_vad_collocate_rows",
            "context_rows",
            "documents",
            "analysis_units",
            "years",
        ],
        "value": [
            len(vad_matches),
            len(context_coverage),
            context_coverage["doc_id"].nunique(),
            ", ".join(observed_units),
            f"{min(observed_years)}-{max(observed_years)}",
        ],
    }
)
handoff_summary


## Annual Arousal Index

Annual arousal is the weighted mean of all matched NRC-VAD collocate occurrences for each analysis unit and publication year. Coverage is reported alongside the index so sparse or unstable unit-years are not over-interpreted.


In [ ]:
coverage = (
    context_coverage.groupby(["lsc_year", "analysis_unit"], as_index=False)
    .agg(
        context_rows=("context_row_id", "size"),
        documents=("doc_id", "nunique"),
        candidate_collocate_tokens=("candidate_collocate_tokens", "sum"),
        matched_vad_units_coverage=("matched_vad_units", "sum"),
        matched_token_positions=("matched_token_positions", "sum"),
        contexts_with_vad_match=("has_vad_match", "sum"),
    )
)
coverage["matched_token_coverage"] = coverage["matched_token_positions"] / coverage["candidate_collocate_tokens"].replace(0, np.nan)
coverage["context_match_coverage"] = coverage["contexts_with_vad_match"] / coverage["context_rows"].replace(0, np.nan)

annual_arousal = (
    vad_matches.groupby(["lsc_year", "analysis_unit", "term_role", "target_group"], as_index=False)
    .agg(
        arousal_mean=("arousal", "mean"),
        arousal_sd=("arousal", "std"),
        valence_mean_for_reference=("valence", "mean"),
        dominance_mean_for_reference=("dominance", "mean"),
        matched_vad_units=("arousal", "size"),
        unique_collocates=("collocate", "nunique"),
        documents_with_matches=("doc_id", "nunique"),
    )
)
annual_arousal = annual_arousal.merge(coverage, on=["lsc_year", "analysis_unit"], how="left")
annual_arousal = annual_arousal.sort_values(["analysis_unit", "lsc_year"]).reset_index(drop=True)

annual_arousal.head(10)


## Bootstrap Confidence Intervals

Uncertainty is estimated by resampling documents within each analysis-unit year. This keeps the Intensity uncertainty contract parallel to Sentiment and avoids treating collocates from the same document as independent observations.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_records: list[dict[str, object]] = []

for (year, unit), group in vad_matches.groupby(["lsc_year", "analysis_unit"], sort=True):
    document_ids = group["doc_id"].drop_duplicates().to_numpy()
    document_arousal = [group.loc[group["doc_id"] == doc_id, "arousal"].to_numpy() for doc_id in document_ids]
    bootstrap_values = []
    for _ in range(BOOTSTRAP_REPETITIONS):
        sampled_indices = rng.integers(0, len(document_arousal), len(document_arousal))
        sampled_scores = np.concatenate([document_arousal[index] for index in sampled_indices])
        bootstrap_values.append(float(np.mean(sampled_scores)))
    bootstrap_records.append(
        {
            "lsc_year": int(year),
            "analysis_unit": unit,
            "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
            "bootstrap_unit": "doc_id",
            "arousal_bootstrap_mean": float(np.mean(bootstrap_values)),
            "arousal_ci_low": float(np.quantile(bootstrap_values, 0.025)),
            "arousal_ci_high": float(np.quantile(bootstrap_values, 0.975)),
        }
    )

bootstrap_arousal = pd.DataFrame(bootstrap_records)
annual_arousal = annual_arousal.merge(bootstrap_arousal, on=["lsc_year", "analysis_unit"], how="left")
annual_arousal.head(10)


## Diagnostics

These diagnostics flag sparse coverage and possible concentration problems. They are not exclusion rules; they indicate which years or units need closer interpretation when reading the arousal trajectory.


In [ ]:
collocate_counts = (
    vad_matches.groupby(["lsc_year", "analysis_unit", "collocate"], as_index=False)
    .agg(
        occurrences=("collocate", "size"),
        arousal=("arousal", "mean"),
    )
)
collocate_totals = collocate_counts.groupby(["lsc_year", "analysis_unit"])["occurrences"].transform("sum")
collocate_counts["occurrence_share"] = collocate_counts["occurrences"] / collocate_totals
collocate_counts["weighted_arousal_contribution"] = collocate_counts["occurrences"] * collocate_counts["arousal"]

top_collocates = (
    collocate_counts.sort_values(["lsc_year", "analysis_unit", "occurrences"], ascending=[True, True, False])
    .groupby(["lsc_year", "analysis_unit"], as_index=False)
    .head(15)
    .reset_index(drop=True)
)

concentration = (
    collocate_counts.sort_values(["lsc_year", "analysis_unit", "occurrences"], ascending=[True, True, False])
    .groupby(["lsc_year", "analysis_unit"], as_index=False)
    .head(5)
    .groupby(["lsc_year", "analysis_unit"], as_index=False)
    .agg(top5_collocate_share=("occurrence_share", "sum"))
)

coverage_for_flags = annual_arousal.merge(concentration, on=["lsc_year", "analysis_unit"], how="left")
flag_rows = []
for row in coverage_for_flags.itertuples(index=False):
    flags = []
    if row.context_rows < 100:
        flags.append("low_context_rows_lt_100")
    if row.documents < 50:
        flags.append("low_documents_lt_50")
    if row.context_match_coverage < 0.90:
        flags.append("low_context_vad_match_coverage_lt_0_90")
    if row.matched_token_coverage < 0.70:
        flags.append("low_token_vad_coverage_lt_0_70")
    if pd.notna(row.top5_collocate_share) and row.top5_collocate_share > 0.35:
        flags.append("top5_collocate_share_gt_0_35")
    if flags:
        flag_rows.append(
            {
                "lsc_year": row.lsc_year,
                "analysis_unit": row.analysis_unit,
                "flags": ";".join(flags),
                "context_rows": row.context_rows,
                "documents": row.documents,
                "context_match_coverage": row.context_match_coverage,
                "matched_token_coverage": row.matched_token_coverage,
                "top5_collocate_share": row.top5_collocate_share,
            }
        )

audit_flag_columns = [
    "lsc_year",
    "analysis_unit",
    "flags",
    "context_rows",
    "documents",
    "context_match_coverage",
    "matched_token_coverage",
    "top5_collocate_share",
]
audit_flags = pd.DataFrame(flag_rows, columns=audit_flag_columns)
coverage_for_flags.sort_values(["analysis_unit", "lsc_year"]).head(10)


## Save Tables

The main annual table is the handoff for downstream synthesis. Coverage, top-collocate, and audit-flag tables remain separate so interpretive diagnostics are easy to inspect without cluttering the main index.


In [ ]:
annual_arousal_path = OUTPUT_DIR / "lsc_intensity_annual_arousal.csv"
coverage_path = OUTPUT_DIR / "lsc_intensity_coverage.csv"
top_collocates_path = OUTPUT_DIR / "lsc_intensity_top_collocates.csv"
audit_flags_path = OUTPUT_DIR / "lsc_intensity_audit_flags.csv"

annual_arousal.to_csv(annual_arousal_path, index=False)
coverage_for_flags.to_csv(coverage_path, index=False)
top_collocates.to_csv(top_collocates_path, index=False)
audit_flags.to_csv(audit_flags_path, index=False)

pd.DataFrame(
    {
        "output": ["annual_arousal", "coverage", "top_collocates", "audit_flags"],
        "path": [annual_arousal_path, coverage_path, top_collocates_path, audit_flags_path],
        "rows": [len(annual_arousal), len(coverage_for_flags), len(top_collocates), len(audit_flags)],
    }
)


## Arousal Trajectories

Targets and baselines are plotted in separate panels to keep the comparison readable. The ribbons show document-level bootstrap confidence intervals.


In [ ]:
def style_axis(ax: plt.Axes) -> None:
    ax.set_xlim(min(EXPECTED_YEARS) - 0.25, max(EXPECTED_YEARS) + 0.25)
    ax.set_xticks(EXPECTED_YEARS)
    ax.tick_params(axis="x", rotation=45)
    ax.set_xlabel("Publication year")
    ax.set_ylabel("Mean NRC-VAD arousal")


fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), sharey=True, constrained_layout=True)
for ax, units, title in zip(axes, [TARGET_UNITS, BASELINE_UNITS], ["Target groups", "Baseline terms"]):
    for unit in units:
        unit_data = annual_arousal[annual_arousal["analysis_unit"] == unit].sort_values("lsc_year")
        years = unit_data["lsc_year"].to_numpy(dtype=float)
        mean = unit_data["arousal_mean"].to_numpy(dtype=float)
        ci_low = unit_data["arousal_ci_low"].to_numpy(dtype=float)
        ci_high = unit_data["arousal_ci_high"].to_numpy(dtype=float)
        ax.plot(
            years,
            mean,
            label=unit,
            color=UNIT_COLOURS[unit],
            marker=UNIT_MARKERS[unit],
            linewidth=2.2,
            markersize=5.5,
        )
        ax.fill_between(years, ci_low, ci_high, color=UNIT_COLOURS[unit], alpha=0.15, linewidth=0)
    ax.set_title(title)
    style_axis(ax)
    ax.legend(loc="best")

fig.suptitle("Annual affective arousal of local VAD collocates", fontsize=13, y=1.03)
trajectory_png = FIGURE_DIR / "lsc_intensity_arousal_trajectories.png"
trajectory_pdf = FIGURE_DIR / "lsc_intensity_arousal_trajectories.pdf"
fig.savefig(trajectory_png, dpi=300, bbox_inches="tight")
fig.savefig(trajectory_pdf, bbox_inches="tight")


## Coverage Overview

Coverage is high when most target-window contexts produce at least one NRC-VAD match and when a large share of candidate collocate token positions are matched. These diagnostics should be checked before interpreting any sharp year-to-year movement.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0), sharey=True, constrained_layout=True)
for ax, metric, title in zip(
    axes,
    ["context_match_coverage", "matched_token_coverage"],
    ["Contexts with any VAD match", "Candidate token positions matched"],
):
    for unit in EXPECTED_UNITS:
        unit_data = coverage_for_flags[coverage_for_flags["analysis_unit"] == unit].sort_values("lsc_year")
        ax.plot(
            unit_data["lsc_year"],
            unit_data[metric],
            label=unit,
            color=UNIT_COLOURS[unit],
            marker=UNIT_MARKERS[unit],
            linewidth=2.0,
            markersize=5,
        )
    ax.set_title(title)
    ax.set_xlim(min(EXPECTED_YEARS) - 0.25, max(EXPECTED_YEARS) + 0.25)
    ax.set_xticks(EXPECTED_YEARS)
    ax.tick_params(axis="x", rotation=45)
    ax.set_xlabel("Publication year")
    ax.set_ylabel("Coverage rate")
    ax.set_ylim(0, 1.05)
axes[1].legend(loc="lower right", ncol=1)

fig.suptitle("NRC-VAD coverage for Intensity collocate windows", fontsize=13, y=1.03)
coverage_png = FIGURE_DIR / "lsc_intensity_coverage.png"
coverage_pdf = FIGURE_DIR / "lsc_intensity_coverage.pdf"
fig.savefig(coverage_png, dpi=300, bbox_inches="tight")
fig.savefig(coverage_pdf, bbox_inches="tight")


## Handoff Summary

The annual arousal table is ready for later integrated synthesis. Any audit flags below should be reviewed before using affected unit-years in substantive interpretation.


In [ ]:
summary = {
    "annual_rows": len(annual_arousal),
    "expected_annual_rows": len(EXPECTED_YEARS) * len(EXPECTED_UNITS),
    "audit_flag_rows": len(audit_flags),
    "min_context_match_coverage": float(coverage_for_flags["context_match_coverage"].min()),
    "min_matched_token_coverage": float(coverage_for_flags["matched_token_coverage"].min()),
    "max_top5_collocate_share": float(coverage_for_flags["top5_collocate_share"].max()),
}
if summary["annual_rows"] != summary["expected_annual_rows"]:
    raise RuntimeError(f"Expected {summary['expected_annual_rows']} annual rows, found {summary['annual_rows']}.")
summary
